# 02 — Analyse des Résultats Expérimentaux

Ce notebook produit l'ensemble des visualisations comparatives.
Il charge tous les JSON de `results/runs/` et génère :

1. **Tableau agrégé** — F1 macro moyen ± écart-type sur les 5 seeds
2. **Courbes d'apprentissage** — F1 en fonction du nombre d'exemples/classe
3. **Heatmap per-classe** — quelles catégories sont difficiles ?
4. **Comparaison few-shot** — zero-shot Ministral vs 8-shot SetFit
5. **Système hybride** — trade-off F1 / coût / taux d'escalade
6. **Trade-off final** — qualité vs coût par modèle

> **Prérequis :** avoir exécuté au moins `run_baseline.py` et `run_camembert.py`

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.evaluation.results import load_all_runs, aggregate_results, save_aggregated

# Palette et styles cohérents
TEMPLATE = 'plotly_white'
MODEL_COLORS = {
    'tfidf_lr':                  '#636EFA',
    'camembert':                 '#EF553B',
    'setfit_camembert':          '#00CC96',
    'setfit_modernbert_en':      '#AB63FA',
    'setfit_modernbert_ml':      '#FFA15A',
    'setfit_mmbert':             '#FF97FF',
    'ministral':                 '#19D3F3',
    'hybrid_setfit_ministral':   '#FF6692',
}
MODEL_LABELS = {
    'tfidf_lr':                  'TF-IDF + LR',
    'camembert':                 'CamemBERT',
    'setfit_camembert':          'SetFit (mpnet-multilingual)',
    'setfit_modernbert_en':      'SetFit (ModernBERT-EN)',
    'setfit_modernbert_ml':      'SetFit (ModernBERT-ML)',
    'setfit_mmbert':             'SetFit (mmBERT)',
    'ministral':                 'Ministral 8B',
    'hybrid_setfit_ministral':   'Hybride (SetFit + Ministral)',
}

In [2]:
df = load_all_runs()
df['model_label'] = df['model'].map(MODEL_LABELS).fillna(df['model'])

print(f"{len(df)} runs chargés")
print(f"Modèles : {sorted(df['model'].unique())}")
print(f"Datasets : {df['dataset'].unique()}")
print(f"Régimes : {sorted(df['regime'].unique())}")

383 runs chargés
Modèles : ['camembert', 'hybrid_setfit_ministral', 'ministral', 'setfit_camembert', 'setfit_mmbert', 'setfit_modernbert_en', 'tfidf_lr']
Datasets : <ArrowStringArray>
['emails', 'newsgroups']
Length: 2, dtype: str
Régimes : ['16', '32', '64', '8', 'few_shot_8', 'full', 'zero_shot']


---
## 1. Tableau agrégé

Moyenne et écart-type du F1 macro sur les 5 seeds, pour chaque combinaison
(modèle × dataset × régime). Un écart-type élevé en few-shot indique
une forte sensibilité au choix des exemples d'entraînement.

In [3]:
agg = aggregate_results(df)
save_aggregated(agg)

# Tri d'abord (sur la colonne numérique), puis mise en forme pourcentage
agg_sorted = agg.sort_values(['dataset', 'f1_macro_mean'], ascending=[True, False])

display_agg = agg_sorted.copy()
display_agg['F1 moyen'] = display_agg['f1_macro_mean'].map('{:.1%}'.format)
display_agg['± écart-type'] = display_agg['f1_macro_std'].map('{:.1%}'.format)
display_agg['Accuracy'] = display_agg['accuracy_mean'].map('{:.1%}'.format)
display_agg = display_agg[['model', 'dataset', 'regime', 'F1 moyen', '± écart-type', 'Accuracy']]
display_agg

,model,dataset,regime,F1 moyen,± écart-type,Accuracy
14,hybrid_setfit_ministral,emails,full,99.3%,0.2%,99.3%
43,tfidf_lr,emails,full,99.2%,0.0%,99.2%
23,setfit_camembert,emails,full,99.2%,0.0%,99.2%
33,setfit_mmbert,emails,full,98.9%,0.1%,98.9%
4,camembert,emails,full,98.6%,0.0%,98.6%
41,tfidf_lr,emails,64,98.2%,0.7%,98.2%
31,setfit_mmbert,emails,64,97.9%,0.2%,97.9%
12,hybrid_setfit_ministral,emails,64,97.8%,0.6%,97.8%
21,setfit_camembert,emails,64,97.8%,0.5%,97.8%
30,setfit_mmbert,emails,32,97.2%,0.8%,97.2%


---
## 2. Courbes d'apprentissage

Chaque courbe montre l'évolution du F1 macro en fonction du nombre d'exemples
par classe. La bande colorée représente ± 1 écart-type sur les 5 seeds.

**Lecture :** plus une courbe monte vite (vers la gauche), mieux le modèle
apprend avec peu de données — c'est l'avantage clé de SetFit et Ministral.

In [4]:
def plot_learning_curves(dataset_name: str) -> go.Figure:
    """Courbes d'apprentissage avec bandes de confiance pour un dataset."""
    sub = df[
        (df['dataset'] == dataset_name) &
        df['regime'].astype(str).str.isdigit()
    ].copy()
    sub['regime'] = sub['regime'].astype(int)

    # Exclure les modèles Ministral (pas de few-shot classique)
    sub = sub[~sub['model'].str.startswith('ministral')]
    sub = sub[~sub['model'].str.startswith('hybrid')]

    curve = sub.groupby(['model', 'regime'])['f1_macro'].agg(['mean', 'std']).reset_index()
    curve['std'] = curve['std'].fillna(0)

    fig = go.Figure()

    for model in curve['model'].unique():
        mc = curve[curve['model'] == model].sort_values('regime')
        color = MODEL_COLORS.get(model, '#888')
        label = MODEL_LABELS.get(model, model)

        # Bande de confiance (± std)
        fig.add_trace(go.Scatter(
            x=list(mc['regime']) + list(mc['regime'])[::-1],
            y=list(mc['mean'] + mc['std']) + list(mc['mean'] - mc['std'])[::-1],
            fill='toself', fillcolor=color, opacity=0.15,
            line=dict(color='rgba(0,0,0,0)'), showlegend=False, hoverinfo='skip',
        ))

        # Courbe principale
        fig.add_trace(go.Scatter(
            x=mc['regime'], y=mc['mean'],
            mode='lines+markers',
            name=label,
            line=dict(color=color, width=2),
            marker=dict(size=8),
            error_y=dict(type='data', array=mc['std'], visible=True, color=color, thickness=1),
        ))

    # Ligne Ministral zero-shot si disponible
    m_zero = df[(df['dataset'] == dataset_name) & (df['regime'] == 'zero_shot')]
    if not m_zero.empty:
        f1_zs = m_zero['f1_macro'].mean()
        fig.add_hline(
            y=f1_zs, line_dash='dot', line_color=MODEL_COLORS.get('ministral', '#19D3F3'),
            annotation_text=f'Ministral zero-shot ({f1_zs:.0%})',
            annotation_position='top right',
        )

    dataset_label = '20 Newsgroups' if dataset_name == 'newsgroups' else 'Emails sportifs'
    fig.update_layout(
        title=f'Courbes d\'apprentissage — {dataset_label}',
        xaxis=dict(type='log', title='Exemples par classe (échelle log)'),
        yaxis=dict(title='F1 macro', range=[0, 1], tickformat='.0%'),
        template=TEMPLATE,
        height=500,
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    return fig


plot_learning_curves('newsgroups').show()

In [5]:
plot_learning_curves('emails').show()

---
## 3. Comparaison en régime full data

Les barres montrent le F1 macro avec toutes les données disponibles.
C'est le plafond de performance de chaque modèle entraînable.

In [6]:
full_df = df[df['regime'] == 'full'].copy()
if not full_df.empty:
    bar = full_df.groupby(['model', 'dataset'])['f1_macro'].mean().reset_index()
    bar['model_label'] = bar['model'].map(MODEL_LABELS).fillna(bar['model'])
    bar['color'] = bar['model'].map(MODEL_COLORS)
    bar['dataset_label'] = bar['dataset'].map(
        {'newsgroups': '20 Newsgroups', 'emails': 'Emails sportifs'}
    )

    fig = px.bar(
        bar, x='f1_macro', y='model_label', color='dataset_label',
        barmode='group', orientation='h',
        title='F1 macro en régime full data — comparaison des modèles',
        labels={'f1_macro': 'F1 macro', 'model_label': '', 'dataset_label': 'Dataset'},
        template=TEMPLATE, text_auto='.1%',
        color_discrete_sequence=['#636EFA', '#00CC96'],
    )
    fig.update_layout(height=400, xaxis=dict(tickformat='.0%', range=[0, 1]))
    fig.show()

---
## 3bis. Comparaison des backbones SetFit — spécialisation vs modernité

Observation centrale de la note. On compare les quatre backbones de SetFit :

| Backbone | Spécialisation FR | Modernité archi. | Multilingue |
|----------|:---:|:---:|:---:|
| **mpnet-multilingual** | ✓ (via multilingue) | ✗ (2021) | ✓ |
| **ModernBERT-EN** | ✗ (anglais) | ✓ (2024) | ✗ |
| **ModernBERT-ML** (e5) | ✓ (via multilingue) | ✓ | ✓ |
| **mmBERT** | ✓ (1800+ langues) | ✓ (2025) | ✓✓ |

mmBERT est le seul à combiner **modernité architecturale** (base ModernBERT,
Flash Attention 2) **et couverture multilingue native** — il tranche donc
directement la question posée dans la note.

In [7]:
# Comparaison directe des backbones SetFit sur les courbes d'apprentissage
# (uniquement les modèles setfit_*, par dataset)
setfit_only = df[
    df['model'].str.startswith('setfit_') &
    df['regime'].astype(str).str.isdigit()
].copy()
setfit_only['regime'] = setfit_only['regime'].astype(int)

if not setfit_only.empty:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['20 Newsgroups', 'Emails sportifs'],
        shared_yaxes=True,
    )

    for col_idx, ds in enumerate(['newsgroups', 'emails'], 1):
        sub = setfit_only[setfit_only['dataset'] == ds]
        curve = sub.groupby(['model', 'regime'])['f1_macro'].agg(['mean', 'std']).reset_index()
        curve['std'] = curve['std'].fillna(0)

        for model in curve['model'].unique():
            mc = curve[curve['model'] == model].sort_values('regime')
            color = MODEL_COLORS.get(model, '#888')
            fig.add_trace(
                go.Scatter(
                    x=mc['regime'], y=mc['mean'],
                    mode='lines+markers',
                    name=MODEL_LABELS.get(model, model),
                    line=dict(color=color, width=2),
                    marker=dict(size=7),
                    error_y=dict(type='data', array=mc['std'], visible=True,
                                 color=color, thickness=1),
                    showlegend=(col_idx == 1),
                    legendgroup=model,
                ),
                row=1, col=col_idx
            )

    fig.update_xaxes(type='log', title='Exemples/classe')
    fig.update_yaxes(range=[0, 1], tickformat='.0%', title='F1 macro', row=1, col=1)
    fig.update_layout(
        title='Comparaison des backbones SetFit — courbes d\'apprentissage',
        height=480, template=TEMPLATE,
        legend=dict(orientation='h', yanchor='bottom', y=-0.25),
    )
    fig.show()
else:
    print("Pas encore de résultats SetFit.")

---
## 4. Analyse few-shot — quelle stratégie pour un club sans données ?

Comparaison directe à 8 exemples par classe : zero-shot Ministral, 8-shot SetFit
et 8-shot TF-IDF. C'est le scénario clé pour un nouveau club.

In [8]:
# Extrait les résultats à 8 exemples + zero-shot Ministral + few-shot Ministral
few = df[df['regime'].isin(['8', 'zero_shot', 'few_shot_8'])].copy()

few_agg = few.groupby(['model', 'dataset', 'regime'])['f1_macro'].agg(
    mean='mean', std='std'
).reset_index()
few_agg['model_label'] = few_agg['model'].map(MODEL_LABELS).fillna(few_agg['model'])
few_agg['std'] = few_agg['std'].fillna(0)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['20 Newsgroups', 'Emails sportifs'])

for col_idx, ds in enumerate(['newsgroups', 'emails'], 1):
    sub = few_agg[few_agg['dataset'] == ds].sort_values('mean', ascending=True)
    fig.add_trace(
        go.Bar(
            y=sub['model_label'],
            x=sub['mean'],
            error_x=dict(type='data', array=sub['std']),
            orientation='h',
            marker_color=[MODEL_COLORS.get(m, '#888') for m in sub['model']],
            text=[f"{v:.0%}" for v in sub['mean']],
            textposition='outside',
            showlegend=False,
        ),
        row=1, col=col_idx
    )

fig.update_xaxes(tickformat='.0%', range=[0, 1])
fig.update_layout(
    title='Comparaison few-shot — 8 exemples/classe (scénario nouveau club)',
    height=450, template=TEMPLATE,
)
fig.show()

---
## 5. Variance entre seeds

La variance mesure la stabilité du modèle : un modèle avec une variance élevée
est risqué en production car ses performances dépendent des données initiales.

In [9]:
# Variance par régime et modèle (emails seulement pour la lisibilité)
var_df = df[
    (df['dataset'] == 'emails') &
    df['regime'].astype(str).str.isdigit() &
    ~df['model'].str.startswith('ministral') &
    ~df['model'].str.startswith('hybrid')
].copy()
var_df['regime'] = var_df['regime'].astype(int)

var_agg = var_df.groupby(['model', 'regime'])['f1_macro'].std().reset_index()
var_agg.columns = ['model', 'regime', 'std_f1']
var_agg['model_label'] = var_agg['model'].map(MODEL_LABELS).fillna(var_agg['model'])

fig = px.line(
    var_agg, x='regime', y='std_f1', color='model_label',
    markers=True,
    color_discrete_map={v: MODEL_COLORS.get(k, '#888') for k, v in MODEL_LABELS.items()},
    title='Écart-type du F1 selon le régime — Emails sportifs',
    labels={'regime': 'Exemples par classe', 'std_f1': 'Écart-type F1', 'model_label': 'Modèle'},
    template=TEMPLATE,
)
fig.update_xaxes(type='log')
fig.update_yaxes(tickformat='.1%')
fig.update_layout(height=400)
fig.show()

---
## 6. Système hybride — trade-off qualité / coût / escalade

Pour chaque seuil τ, on mesure le F1 du système hybride, le taux d'escalade
(part des mails routés vers Ministral) et le coût API moyen par mail.

In [10]:
hybrid = df[df['model'] == 'hybrid_setfit_ministral'].copy()

if hybrid.empty:
    print("Pas encore de résultats hybrides. Lance run_hybrid.py d'abord.")
else:
    hybrid['threshold'] = hybrid['threshold'].astype(float)
    h_agg = hybrid.groupby(['dataset', 'threshold']).agg(
        f1_macro=('f1_macro', 'mean'),
        escalation_rate=('escalation_rate', 'mean'),
        avg_cost=('avg_cost_per_mail_usd', 'mean')
    ).reset_index()
    h_agg['dataset_label'] = h_agg['dataset'].map(
        {'newsgroups': '20 Newsgroups', 'emails': 'Emails sportifs'}
    )

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['F1 macro vs seuil τ', 'Taux d\'escalade vs seuil τ']
    )
    colors = {'20 Newsgroups': '#636EFA', 'Emails sportifs': '#00CC96'}

    for ds_label in h_agg['dataset_label'].unique():
        sub = h_agg[h_agg['dataset_label'] == ds_label]
        c = colors.get(ds_label, '#888')
        fig.add_trace(go.Scatter(x=sub['threshold'], y=sub['f1_macro'],
                                  mode='lines+markers', name=ds_label,
                                  line=dict(color=c), marker=dict(size=8)),
                      row=1, col=1)
        fig.add_trace(go.Scatter(x=sub['threshold'], y=sub['escalation_rate'],
                                  mode='lines+markers', name=ds_label,
                                  line=dict(color=c), marker=dict(size=8),
                                  showlegend=False),
                      row=1, col=2)

    fig.update_yaxes(tickformat='.0%', row=1, col=1)
    fig.update_yaxes(tickformat='.0%', row=1, col=2)
    fig.update_layout(
        title='Système hybride — impact du seuil de confiance τ',
        height=400, template=TEMPLATE,
    )
    fig.show()

In [11]:
if not hybrid.empty:
    # Scatter trade-off qualité vs coût
    fig = px.scatter(
        h_agg, x='avg_cost', y='f1_macro',
        color='dataset_label', size='escalation_rate',
        text='threshold', size_max=30,
        title='Trade-off qualité vs coût moyen par mail (hybride)',
        labels={
            'avg_cost': 'Coût moyen par mail (USD)',
            'f1_macro': 'F1 macro',
            'dataset_label': 'Dataset',
        },
        color_discrete_sequence=['#636EFA', '#00CC96'],
        template=TEMPLATE,
    )
    fig.update_traces(textposition='top center')
    fig.update_yaxes(tickformat='.0%')
    fig.update_layout(height=450)
    fig.add_annotation(
        text="← Idéal : F1 élevé, coût faible",
        xref='paper', yref='paper', x=0.01, y=0.99,
        showarrow=False, font=dict(color='gray')
    )
    fig.show()

---
## 7. Trade-off global — qualité vs latence vs coût

Vue synthétique positionnant chaque stratégie selon ses trois dimensions clés.
C'est la base visuelle de la matrice de décision produit (notebook 04).

In [12]:
# Construction manuelle du tableau de synthèse (à compléter avec les résultats réels)
# Remplir f1_emails et inference_ms après avoir lancé tous les scripts

full_means = (
    df[df['regime'] == 'full']
    .groupby(['model', 'dataset'])['f1_macro'].mean()
    .unstack(fill_value=float('nan'))
    .reset_index()
)

latency = (
    df[df['regime'] == 'full']
    .groupby('model')['inference_time_ms'].mean()
    .reset_index()
) if 'inference_time_ms' in df.columns else pd.DataFrame()

print("F1 macro en régime full data par modèle et dataset :")
print(full_means.to_string(index=False))

if not latency.empty:
    print("\nLatence d'inférence moyenne (ms/mail) :")
    print(latency.to_string(index=False))

F1 macro en régime full data par modèle et dataset :
                  model   emails  newsgroups
              camembert 0.986109    0.786699
hybrid_setfit_ministral 0.993235         NaN
       setfit_camembert 0.991634    0.837131
          setfit_mmbert 0.989413         NaN
   setfit_modernbert_en      NaN    0.804001
               tfidf_lr 0.991696    0.799809

Latence d'inférence moyenne (ms/mail) :
                  model  inference_time_ms
              camembert            3.50150
hybrid_setfit_ministral                NaN
       setfit_camembert            1.71005
          setfit_mmbert            2.28088
   setfit_modernbert_en            2.30828
               tfidf_lr            0.12623
